# LimiX-2 — prueba en el fold 0
Modelo fundacional tabular (Stable AI / Tsinghua, sept 2026; primero en TabArena). Se mide accuracy, tiempo y memoria de GPU con contextos de 10.000, 25.000 y 56.000 filas sobre el fold 0 de validacion, con las mismas features que TabPFN.

In [ ]:
import subprocess, sys, os, time
print(sys.version)
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
!git clone -q https://github.com/limix-ldm-ai/LimiX.git /kaggle/working/LimiX
!cd /kaggle/working/LimiX && pip install -q -e . 2>&1 | tail -3

In [ ]:
import glob, numpy as np, pandas as pd, torch
os.environ.update(RANK="0", WORLD_SIZE="1", MASTER_ADDR="127.0.0.1", MASTER_PORT="29500")
sys.path.insert(0, "/kaggle/working/LimiX")
from inference.predictor import LimiXPredictor
from huggingface_hub import hf_hub_download
CKPT = hf_hub_download(repo_id="stable-ai/LimiX-2", filename="LimiX-2.ckpt", local_dir="/kaggle/working/cache")
CFG = "/kaggle/working/LimiX/config/cls_default_noretrieval_v2.json"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
DATA = os.path.dirname(glob.glob("/kaggle/input/**/fold0_tr.parquet", recursive=True)[0])
y = np.load(f"{DATA}/y.npy")
from sklearn.model_selection import StratifiedKFold
a, b = next(iter(StratifiedKFold(5, shuffle=True, random_state=0).split(y, y)))   # = features.cv_folds, fold 0
A, B = pd.read_parquet(f"{DATA}/fold0_tr.parquet"), pd.read_parquet(f"{DATA}/fold0_va.parquet")

def prepare(A, B):
    """Categoricas -> codigos enteros con el mismo diccionario en A y B (igual que en TabPFN)."""
    A, B = A.copy(), B.copy()
    cats = [c for c in A.columns if str(A[c].dtype) == "category"]
    for c in cats:
        dic = pd.Index(pd.concat([A[c].astype(str), B[c].astype(str)]).unique())
        A[c], B[c] = dic.get_indexer(A[c].astype(str)), dic.get_indexer(B[c].astype(str))
    return A.astype("float32").values, B.astype("float32").values, [A.columns.get_loc(c) for c in cats]

XA, XB, cat_idx = prepare(A, B)
ya, yb = y[a], y[b]
print(XA.shape, XB.shape, "categoricas:", len(cat_idx))

In [ ]:
clf = LimiXPredictor(device=torch.device("cuda"), model_path=CKPT, inference_config=CFG, categorical_features_indices=cat_idx)
out = {}
for n_ctx in [10000, 25000, 56000]:
    rng = np.random.default_rng(0)
    sel = np.sort(rng.choice(len(XA), min(n_ctx, len(XA)), replace=False))
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    try:
        p = np.concatenate([clf.predict(XA[sel], ya[sel], XB[s:s + 2000], task_type="Classification")[:, 1] for s in range(0, len(XB), 2000)])
    except Exception as e:
        print(f"contexto {n_ctx}: fallo -> {type(e).__name__}: {str(e)[:300]}", flush=True); torch.cuda.empty_cache(); continue
    out[f"valid0_ctx{n_ctx}"] = p
    np.savez("limix_fold0.npz", **out)
    print(f"contexto {n_ctx}: accuracy {((p > .5) == yb).mean():.4f} | {(time.time() - t0) / 60:.1f} min | memoria pico {torch.cuda.max_memory_allocated() / 2**30:.1f} GB", flush=True)